# Try the pruned models on real pictures

Live-demo notebook: loads the **baseline** model plus the **quantum (QAOA)
pruned** model and the **classical (greedy) pruned** model side by side, and
lets you run any street-view photo through all three to compare predictions,
confidence, and inference speed.

Masks are read directly from `qubo_outputs/qaoa_ranked_masks.csv` (top row)
and `qubo_outputs/classical_pruning_result.json`, so this notebook always
reflects whatever the pipeline most recently produced — re-run
`qubo_hamiltonian.py` / `qaoa_rank_masks.ipynb` / `classical_pruning.ipynb`
at a new target compression, then re-run this notebook to see the new masks
in action.

**Two ways to pick an image:**
1. Browse real held-out test photos (guaranteed to work, no extra setup) — Step 5.
2. Upload your own photo with the widget in Step 6 (works in a live Jupyter
   session; the model still forces a prediction into one of its 15 known
   Canadian cities even if your photo isn't one of them — that's expected,
   not a bug).

## Step 1 — Imports and configuration

In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
import torch.nn as nn
from PIL import Image
import matplotlib.pyplot as plt
from torchvision.transforms import v2

import timm
from datasets import load_dataset
from huggingface_hub import hf_hub_download

PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "qubo_outputs"

CLASS_NAMES = [
    "Calgary", "Charlottetown", "Edmonton", "Halifax", "Hamilton",
    "Kitchener-Waterloo", "Montreal", "Ottawa-Gatineau", "Quebec City",
    "Saskatoon", "St Johns", "Toronto", "Vancouver", "Victoria", "Winnipeg",
]

MODEL_NAME = "convnext"
N_SAMPLE_IMAGES = 12

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Step 2 — Shared helpers (transform, model loading, mask application)

Duplicated from `top_k_mask_evaluation.ipynb` / `classical_pruning.ipynb` so
this notebook is self-contained, matching the rest of the project.

In [ ]:
def resize_and_pad(img: Image.Image, target_size=(320, 320)) -> Image.Image:
    img = img.copy()
    img.thumbnail(target_size, Image.Resampling.LANCZOS)
    new_img = Image.new("RGB", target_size, (0, 0, 0))
    left = (target_size[0] - img.size[0]) // 2
    top = (target_size[1] - img.size[1]) // 2
    new_img.paste(img, (left, top))
    return new_img


TRANSFORM = v2.Compose([
    v2.Lambda(lambda img: resize_and_pad(img)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def torch_load_compatible(path: str, device: torch.device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def load_finetuned_model(device: torch.device) -> nn.Module:
    path = hf_hub_download(
        repo_id="canada-guesser/canadian_streetview_cities_models",
        filename="cnn_model/convnext_tiny_set_3_final.bin",
    )
    model = timm.create_model("convnext_tiny", pretrained=False, num_classes=len(CLASS_NAMES))
    checkpoint = torch_load_compatible(path, device)
    state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model


class CandidateBypassWrapper(nn.Module):
    def __init__(self, module: nn.Module):
        super().__init__()
        self.module = module

    def forward(self, x, *args, **kwargs):
        return x


def get_parent_and_child(model: nn.Module, module_name: str):
    parts = module_name.split(".")
    parent = model
    for part in parts[:-1]:
        parent = parent[int(part)] if part.isdigit() else getattr(parent, part)
    return parent, parts[-1]


def get_module_by_name(model: nn.Module, module_name: str) -> nn.Module:
    module = model
    for part in module_name.split("."):
        module = module[int(part)] if part.isdigit() else getattr(module, part)
    return module


def replace_module(model: nn.Module, module_name: str, new_module: nn.Module) -> None:
    parent, child_key = get_parent_and_child(model, module_name)
    if child_key.isdigit():
        parent[int(child_key)] = new_module
    else:
        setattr(parent, child_key, new_module)


def apply_pruning_mask(model: nn.Module, blocks: List[str]) -> None:
    for block_name in blocks:
        block_name = block_name.strip()
        if not block_name:
            continue
        original_module = get_module_by_name(model, block_name)
        replace_module(model, block_name, CandidateBypassWrapper(original_module))


def parse_pruned_blocks(text: str) -> List[str]:
    text = str(text or "").strip()
    return [part.strip() for part in text.split(";") if part.strip()]

## Step 3 — Load the quantum and classical masks (whatever the pipeline last produced)

In [ ]:
import csv

with (OUTPUT_DIR / "qaoa_ranked_masks.csv").open("r", newline="", encoding="utf-8") as f:
    qaoa_rows = list(csv.DictReader(f))
quantum_top_row = sorted(qaoa_rows, key=lambda r: float(r["energy"]))[0]
quantum_pruned_blocks = parse_pruned_blocks(quantum_top_row["pruned_blocks"])

with (OUTPUT_DIR / "classical_pruning_result.json").open("r", encoding="utf-8") as f:
    classical_summary = json.load(f)
classical_pruned_blocks = parse_pruned_blocks(classical_summary["result"]["pruned_blocks"])

with (OUTPUT_DIR / "qubo_metadata.json").open("r", encoding="utf-8") as f:
    target_compression = json.load(f)["target_compression"]

print(f"Target compression: {target_compression:.0%}")
print(f"Quantum (QAOA) mask   : {quantum_top_row['bitstring']} -> {quantum_pruned_blocks}")
print(f"Classical (greedy) mask: {classical_summary['result']['mask']} -> {classical_pruned_blocks}")

## Step 4 — Load the three models once (kept in memory for fast repeated inference)

In [ ]:
print("Loading baseline model...")
baseline_model = load_finetuned_model(device)

print("Loading quantum (QAOA)-pruned model...")
quantum_model = load_finetuned_model(device)
apply_pruning_mask(quantum_model, quantum_pruned_blocks)

print("Loading classical (greedy)-pruned model...")
classical_model = load_finetuned_model(device)
apply_pruning_mask(classical_model, classical_pruned_blocks)

MODELS = {
    "Baseline (unpruned)": baseline_model,
    "Quantum (QAOA) pruned": quantum_model,
    "Classical (greedy) pruned": classical_model,
}

print("\nAll three models loaded and ready.")

## Step 5 — Prediction + side-by-side comparison helpers

In [ ]:
@torch.no_grad()
def predict(image: Image.Image, model: nn.Module) -> Dict[str, Any]:
    x = TRANSFORM(image.convert("RGB")).unsqueeze(0).to(device)

    start = time.perf_counter()
    logits = model(x)
    if hasattr(logits, "logits"):
        logits = logits.logits
    probs = torch.softmax(logits, dim=1)[0]
    inference_ms = (time.perf_counter() - start) * 1000.0

    top_idx = int(torch.argmax(probs).item())

    return {
        "predicted_city": CLASS_NAMES[top_idx],
        "confidence": float(probs[top_idx].item()),
        "inference_ms": inference_ms,
    }


def compare_all(image: Image.Image, true_label: str | None = None) -> None:
    plt.figure(figsize=(4, 4))
    plt.imshow(image.convert("RGB"))
    plt.axis("off")
    title = f"True label: {true_label}" if true_label else "Uploaded image"
    plt.title(title)
    plt.show()

    header = f"{'Model':<26} {'Prediction':<20} {'Confidence':>10} {'Time (ms)':>10}"
    print(header)
    print("-" * len(header))
    for name, model in MODELS.items():
        result = predict(image, model)
        mark = ""
        if true_label is not None:
            mark = "  <-- correct" if result["predicted_city"] == true_label else "  <-- wrong"
        print(
            f"{name:<26} {result['predicted_city']:<20} "
            f"{result['confidence']*100:>9.1f}% {result['inference_ms']:>9.1f}{mark}"
        )

## Step 6 — Browse real held-out test photos

Loads a handful of real test-set images with true labels. Change `INDEX` and
re-run the last cell to flip through them live.

In [ ]:
print("Loading sample test images (first run downloads/caches the dataset)...")
sample_ds = load_dataset(
    "canada-guesser/Canadian-streetview-cities",
    split=f"test[:{N_SAMPLE_IMAGES}]",
)

samples = []
for row in sample_ds:
    img = row["image"]
    if not isinstance(img, Image.Image):
        img = Image.open(img)
    samples.append({"image": img, "true_label": CLASS_NAMES[int(row["label"])]})

print(f"Loaded {len(samples)} sample images. Set INDEX below (0..{len(samples)-1}) and re-run.")

In [ ]:
INDEX = 0  # <-- change this and re-run this cell during the live demo

compare_all(samples[INDEX]["image"], true_label=samples[INDEX]["true_label"])

## Step 7 — Upload your own photo (live Jupyter only)

Requires running this notebook in an actual Jupyter session (not a plain
script) so the upload widget renders. If no file is uploaded (e.g. during an
automated/headless run), this cell safely falls back to sample image 0
instead of erroring.

In [ ]:
import io
import ipywidgets as widgets
from IPython.display import display

uploader = widgets.FileUpload(accept="image/*", multiple=False)
display(uploader)
print("Upload a photo above, then run the next cell.")

In [ ]:
if len(uploader.value) > 0:
    uploaded_file = list(uploader.value.values())[0] if isinstance(uploader.value, dict) else uploader.value[0]
    content = uploaded_file["content"] if isinstance(uploaded_file, dict) else uploaded_file.content
    uploaded_image = Image.open(io.BytesIO(bytes(content)))
    compare_all(uploaded_image, true_label=None)
else:
    print("No file uploaded yet -- showing sample image 0 as a fallback so this cell still runs cleanly.")
    compare_all(samples[0]["image"], true_label=samples[0]["true_label"])